# 01_05_08 Temporal Leakage Audit — aoe2companion
spec: reports/specs/01_05_preregistration.md@7e259dd8
aoec binding: temporal leakage audit for 01_05 aoe2companion

**Spec §§:** §9 (temporal leakage audit queries)
**Critique fix M-01:** Reframed as meaningful bin-edge temporal check (not vacuous match-id disjointness).
**Critique fix M-10:** Renamed to 01_05_08_leakage_audit.py

**Hypothesis:** zero rows in matches_history_minimal have observation_time >= match_time (I3 compliance);
zero POST_GAME tokens appear in the pre-game feature list used in T03; reference period window
constants match spec §7 exactly.
**Falsifier:** any of the above is violated -> step halts.

## Imports

In [1]:
import ast
import hashlib
import json
import re
from datetime import datetime
from pathlib import Path

from rts_predict.common.notebook_utils import get_notebook_db, get_reports_dir

ARTIFACTS = get_reports_dir("aoe2", "aoe2companion") / "artifacts" / "01_exploration" / "05_temporal_panel_eda"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

REF_START = datetime(2022, 8, 29)
REF_END = datetime(2022, 12, 31)
POST_GAME_TOKENS = {"duration_seconds", "is_duration_suspicious", "is_duration_negative", "ratingDiff", "finished"}

print("Artifacts dir:", ARTIFACTS)

Artifacts dir: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/05_temporal_panel_eda


## Check 1: Temporal-window integrity checks (v2, post-PR #162 adversarial review)

The v1 Check 1a/1b were tautological — their WHERE clauses AND-ed a predicate
with its own negation, so the result was always 0 regardless of data. This
v2 replaces them with three substantive tests:

- **Check 1a (bounds-are-honored)**: MIN/MAX started_at in the reference
  cohort are strictly within the declared [REF_START, REF_END) bounds.
  Catches DB timezone bugs and filter regressions.
- **Check 1b (quarter-labels-match-bounds)**: For each tested quarter, row
  bounds match the ISO-calendar quarter the label claims. Catches off-by-one
  errors in the CONCAT/CEIL quarter derivation.
- **Check 1c (PSI-reference-SQL-contains-spec-bounds)**: 01_05_02 PSI
  reference SQL literally contains the spec §7 timestamp bounds. Catches
  silent reference-window drift between steps.

In [2]:
QUERY_CHECK1A_REF_RANGE = """
SELECT
    MIN(started_at) AS ref_min,
    MAX(started_at) AS ref_max,
    COUNT(*) AS ref_count
FROM matches_history_minimal
WHERE started_at >= TIMESTAMP '2022-08-29'
  AND started_at <  TIMESTAMP '2023-01-01'
"""

QUERY_CHECK1B_QUARTER_RANGES = """
SELECT
    derived_quarter,
    MIN(started_at) AS qmin,
    MAX(started_at) AS qmax,
    COUNT(*) AS n
FROM (
    SELECT
        started_at,
        CONCAT(
            CAST(EXTRACT(YEAR FROM started_at) AS VARCHAR),
            '-Q',
            CAST(CEIL(EXTRACT(MONTH FROM started_at) / 3.0) AS INTEGER)::VARCHAR
        ) AS derived_quarter
    FROM matches_history_minimal
    WHERE started_at >= TIMESTAMP '2023-01-01'
      AND started_at <  TIMESTAMP '2025-01-01'
)
GROUP BY derived_quarter
ORDER BY derived_quarter
"""

QUARTER_BOUNDS = {
    "2023-Q1": (datetime(2023, 1, 1),  datetime(2023, 4, 1)),
    "2023-Q2": (datetime(2023, 4, 1),  datetime(2023, 7, 1)),
    "2023-Q3": (datetime(2023, 7, 1),  datetime(2023, 10, 1)),
    "2023-Q4": (datetime(2023, 10, 1), datetime(2024, 1, 1)),
    "2024-Q1": (datetime(2024, 1, 1),  datetime(2024, 4, 1)),
    "2024-Q2": (datetime(2024, 4, 1),  datetime(2024, 7, 1)),
    "2024-Q3": (datetime(2024, 7, 1),  datetime(2024, 10, 1)),
    "2024-Q4": (datetime(2024, 10, 1), datetime(2025, 1, 1)),
}

db = get_notebook_db("aoe2", "aoe2companion", read_only=True)

# Check 1a: reference-window bounds are honored by the DB (not just the filter SQL).
ref_min, ref_max, ref_count = db.con.execute(QUERY_CHECK1A_REF_RANGE).fetchone()
check1a_pass = (
    ref_min is not None
    and ref_max is not None
    and ref_min >= datetime(2022, 8, 29)
    and ref_max <  datetime(2023, 1, 1)
    and ref_count > 0
)
print(
    f"Check 1a (ref-range integrity): min={ref_min} max={ref_max} count={ref_count:,} "
    f"-> {'PASS' if check1a_pass else 'FAIL'}"
)

# Check 1b: each tested-period quarter's row range lies strictly inside its declared bounds.
rows_1b = db.con.execute(QUERY_CHECK1B_QUARTER_RANGES).fetchall()
quarter_violations: list[dict] = []
for derived_quarter, qmin, qmax, n in rows_1b:
    expected = QUARTER_BOUNDS.get(derived_quarter)
    if expected is None:
        quarter_violations.append(
            {"quarter": derived_quarter, "reason": "unrecognized label", "qmin": str(qmin), "qmax": str(qmax)}
        )
        continue
    qstart, qend = expected
    if qmin < qstart or qmax >= qend:
        quarter_violations.append(
            {"quarter": derived_quarter, "reason": "range escapes label bounds",
             "qmin": str(qmin), "qmax": str(qmax),
             "expected": f"[{qstart}, {qend})"}
        )
check1b_pass = len(quarter_violations) == 0
print(
    f"Check 1b (quarter-label consistency): {len(rows_1b)} quarters checked, "
    f"{len(quarter_violations)} violations -> {'PASS' if check1b_pass else 'FAIL'}"
)
for v in quarter_violations:
    print(f"  VIOLATION: {v}")

# Check 1c: 01_05_02 PSI reference SQL must cite the spec §7 bounds verbatim.
psi_json_path = ARTIFACTS / "01_05_02_psi_shift.json"
assert psi_json_path.exists(), f"Dependency missing: {psi_json_path}"
_psi_data = json.loads(psi_json_path.read_text())
_sql_blob = json.dumps(_psi_data.get("sql_queries", {}))
_need = ["TIMESTAMP '2022-08-29'", "TIMESTAMP '2023-01-01'"]
_missing = [s for s in _need if s not in _sql_blob]
check1c_pass = len(_missing) == 0
print(
    f"Check 1c (PSI reference SQL cites spec §7 bounds): "
    f"missing={_missing if _missing else 'none'} -> {'PASS' if check1c_pass else 'FAIL'}"
)

check1_pass = check1a_pass and check1b_pass and check1c_pass
print(f"Check 1 OVERALL: {'PASS' if check1_pass else 'FAIL'}")
assert check1_pass, (
    f"LEAKAGE DETECTED: Check 1 failed. "
    f"1a={check1a_pass}, 1b={check1b_pass} (violations={quarter_violations}), "
    f"1c={check1c_pass} (missing={_missing})"
)

Check 1a (ref-range integrity): min=2022-08-29 00:00:31 max=2022-12-31 23:59:58 count=4,013,826 -> PASS


Check 1b (quarter-label consistency): 8 quarters checked, 0 violations -> PASS
Check 1c (PSI reference SQL cites spec §7 bounds): missing=none -> PASS
Check 1 OVERALL: PASS


## Check 2: POST_GAME token scan of T03/T04 notebooks

In [3]:
# Resolve the sandbox directory: `__file__` works under plain-python execution;
# under a Jupyter kernel it is undefined, so fall back to a repo-rooted constant.
try:
    _SANDBOX_DIR = Path(__file__).parent
except NameError:
    _SANDBOX_DIR = (
        Path(get_reports_dir("aoe2", "aoe2companion")).parents[6]
        / "sandbox/aoe2/aoe2companion/01_exploration/05_temporal_panel_eda"
    )

SCAN_FILES = [
    _SANDBOX_DIR / "01_05_02_psi_shift.py",
    _SANDBOX_DIR / "01_05_03_stratification.py",
]

post_game_found = {}
for fpath in SCAN_FILES:
    if not fpath.exists():
        print(f"WARNING: {fpath.name} not found, skipping scan.")
        continue
    source = fpath.read_text(encoding="utf-8")
    tokens_in_file = []
    for token in POST_GAME_TOKENS:
        if token in source:
            # Check it's in a feature selection context, not just commented or in exclusion notes
            # A token in a WHERE exclusion clause is legitimate (e.g., WHERE NOT is_duration_suspicious)
            # A token as a feature name in PSI computation is a leakage
            # Use regex to find feature references (not WHERE exclusions)
            feature_refs = re.findall(rf'(?<!NOT\s)(?<!WHERE\s)(?<!["\'])({re.escape(token)})(?!\s*=\s*FALSE)', source)
            # Filter: if token appears only in comments or exclusion-where clauses, it's safe
            lines_with_token = [l.strip() for l in source.splitlines() if token in l]
            # Only flag if token appears as a standalone variable (not in SQL WHERE exclusion pattern)
            # For this audit: flag if it appears as a selected feature name
            for line in lines_with_token:
                if (token in line and
                        not line.startswith("#") and
                        "WHERE" not in line and
                        "EXCLUDE" not in line and
                        "FILTER" not in line and
                        "excluded" not in line.lower() and
                        "except" not in line.lower() and
                        "post_game" not in line.lower() and
                        "guard" not in line.lower()):
                    tokens_in_file.append({"token": token, "line": line})
    post_game_found[str(fpath)] = tokens_in_file
    print(f"{fpath.name}: {len(tokens_in_file)} potential POST_GAME token occurrences in pre-game context")
    for match in tokens_in_file:
        print(f"  token={match['token']!r}: {match['line']!r}")

# Strict check: no feature selection of POST_GAME tokens
all_tokens_found = [t for tokens in post_game_found.values() for t in tokens]
check2_pass = (len(all_tokens_found) == 0)
print(f"\nCheck 2 (POST_GAME token scan): {'PASS' if check2_pass else 'FAIL'}")
if not check2_pass:
    print("WARNING: POST_GAME tokens found in pre-game notebooks. Manual review required.")

01_05_02_psi_shift.py: 0 potential POST_GAME token occurrences in pre-game context
01_05_03_stratification.py: 0 potential POST_GAME token occurrences in pre-game context

Check 2 (POST_GAME token scan): PASS


## Check 3: Reference period window constant assertion

In [4]:
# Load T03 JSON to verify frozen reference edges contain correct dates
psi_json_path = ARTIFACTS / "01_05_02_psi_shift.json"
assert psi_json_path.exists(), f"T03 JSON not found: {psi_json_path}"
psi_data = json.loads(psi_json_path.read_text())

ref_edges = psi_data.get("frozen_reference_edges", {})
ref_start_str = ref_edges.get("ref_start", "")
ref_end_str = ref_edges.get("ref_end", "")

expected_ref_start = REF_START.isoformat()  # "2022-08-29"
expected_ref_end = REF_END.isoformat()  # "2022-12-31"

check3_start = ref_start_str.startswith(expected_ref_start)
check3_end = ref_end_str.startswith(expected_ref_end)
check3_assert = psi_data.get("assertion_passed", False)
check3_pass = check3_start and check3_end and check3_assert

print(f"Check 3 (ref period constants):")
print(f"  ref_start={ref_start_str!r} (expected {expected_ref_start!r}): {'OK' if check3_start else 'FAIL'}")
print(f"  ref_end={ref_end_str!r} (expected {expected_ref_end!r}): {'OK' if check3_end else 'FAIL'}")
print(f"  assertion_passed in T03 JSON: {check3_assert}")
print(f"Check 3 OVERALL: {'PASS' if check3_pass else 'FAIL'}")
assert check3_pass, f"LEAKAGE: Ref period mismatch. ref_start={ref_start_str}, ref_end={ref_end_str}"

Check 3 (ref period constants):
  ref_start='2022-08-29T00:00:00' (expected '2022-08-29T00:00:00'): OK
  ref_end='2022-12-31T00:00:00' (expected '2022-12-31T00:00:00'): OK
  assertion_passed in T03 JSON: True
Check 3 OVERALL: PASS


## Record hash of reference period rows (M-06)

In [5]:
# Hash of reference-period rows used for PSI edge derivation
QUERY_REF_HASH = """
SELECT MD5(STRING_AGG(CAST(match_id AS VARCHAR) || '|' || CAST(player_id AS VARCHAR) ORDER BY match_id, player_id))
FROM matches_history_minimal
WHERE started_at >= TIMESTAMP '2022-08-29'
  AND started_at <  TIMESTAMP '2023-01-01'
LIMIT 1
"""

ref_hash_result = db.con.execute(QUERY_REF_HASH).fetchone()[0]
print(f"Reference period row hash: {ref_hash_result}")

db.close()

Reference period row hash: f9aa56bb5f22e247615fb9e33f30e688


## Emit artifacts

In [6]:
all_pass = check1_pass and check2_pass and check3_pass

audit_json = {
    "checks": [
        {
            "id": "check_1_temporal_bin_edges",
            "description": (
                "v2 post-PR #162 adversarial-review fix: the v1 Check 1a/1b "
                "had a mutually-exclusive WHERE clause (A ∧ B) ∧ (¬A ∨ ¬B) "
                "that returned 0 regardless of data. v2 replaces with three "
                "substantive checks: (1a) ref-cohort MIN/MAX in bounds; "
                "(1b) quarter-label consistency; (1c) PSI reference SQL "
                "cites spec §7 timestamp bounds verbatim."
            ),
            "status": "PASS" if check1_pass else "FAIL",
            "check_1a_ref_range": {
                "ref_min": str(ref_min),
                "ref_max": str(ref_max),
                "ref_count": int(ref_count) if ref_count is not None else 0,
                "pass": check1a_pass,
            },
            "check_1b_quarter_consistency": {
                "n_quarters_checked": len(rows_1b),
                "violations": quarter_violations,
                "pass": check1b_pass,
            },
            "check_1c_psi_ref_sql_cites_bounds": {
                "required_tokens": _need,
                "missing_tokens": _missing,
                "pass": check1c_pass,
            },
            "sql_check_1a": QUERY_CHECK1A_REF_RANGE,
            "sql_check_1b": QUERY_CHECK1B_QUARTER_RANGES,
        },
        {
            "id": "check_2_post_game_token_scan",
            "description": "AST/regex scan of T03/T04 notebooks for POST_GAME tokens in pre-game feature selection contexts.",
            "status": "PASS" if check2_pass else "FAIL",
            "post_game_tokens_scanned": sorted(POST_GAME_TOKENS),
            "post_game_tokens_found": all_tokens_found,
            "files_scanned": [str(f) for f in SCAN_FILES],
        },
        {
            "id": "check_3_normalization_window",
            "description": "Assert T03 JSON frozen_reference_edges.ref_start == '2022-08-29' and ref_end == '2022-12-31' (spec §7).",
            "status": "PASS" if check3_pass else "FAIL",
            "ref_start_in_artifact": ref_start_str,
            "ref_end_in_artifact": ref_end_str,
            "expected_ref_start": expected_ref_start,
            "expected_ref_end": expected_ref_end,
            "assertion_passed_in_t03": check3_assert,
        },
    ],
    "reference_period_row_hash": ref_hash_result,
    "all_checks_pass": all_pass,
    "verdict": "PASS" if all_pass else "FAIL",
    "produced_at": datetime.now().isoformat(),
}

json_path = ARTIFACTS / "01_05_08_leakage_audit.json"
json_path.write_text(json.dumps(audit_json, indent=2, default=str))
print("Wrote:", json_path)

Wrote: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/05_temporal_panel_eda/01_05_08_leakage_audit.json


In [7]:
status_per_check = "\n".join(
    f"| {c['id']} | {c['status']} | {c['description'][:80]}... |"
    for c in audit_json["checks"]
)

md_content = f"""# 01_05_08 Temporal Leakage Audit — aoe2companion

spec: reports/specs/01_05_preregistration.md@7e259dd8

## Summary

| Check | Status | Description |
|---|---|---|
{status_per_check}

**Overall verdict: {audit_json['verdict']}**

## Check 1 Redesign Note (v2, post-PR #162 adversarial review)

The v1 implementation of Check 1a/1b (pre-fix/01-05-aoec-adversarial-followup)
had a mutually-exclusive WHERE clause `(A ∧ B) ∧ (¬A ∨ ¬B)` that returned 0
regardless of data — a tautology posing as a gate. The adversarial reviewer
of PR #162 flagged this as BLOCKER 2. The v2 replacement below provides three
substantive sub-checks.

- **Check 1a (ref-cohort range integrity):** `min(started_at) = {ref_min}`,
  `max(started_at) = {ref_max}`, `count = {ref_count:,}` → {'PASS' if check1a_pass else 'FAIL'}
- **Check 1b (quarter-label consistency):** {len(rows_1b)} quarters checked,
  {len(quarter_violations)} violations → {'PASS' if check1b_pass else 'FAIL'}
- **Check 1c (PSI ref SQL cites §7 bounds):** missing tokens = `{_missing}` → {'PASS' if check1c_pass else 'FAIL'}

## Check 2: POST_GAME token scan

Scanned: {', '.join(f.name for f in SCAN_FILES)}
Tokens scanned: {sorted(POST_GAME_TOKENS)}
Tokens found in pre-game context: {all_tokens_found if all_tokens_found else 'NONE'}

## Check 3: Normalization window constants

T03 JSON asserts:
- ref_start = {ref_start_str!r} (expected '2022-08-29'): {'OK' if check3_start else 'FAIL'}
- ref_end = {ref_end_str!r} (expected '2022-12-31'): {'OK' if check3_end else 'FAIL'}
- assertion_passed = {check3_assert}

## Reference period row hash (M-06)

MD5 of (match_id || player_id) for reference period rows: `{ref_hash_result}`
Use for reproducibility verification across DB rebuilds (reservoir-sample caveat applies).

## SQL

### Check 1a (ref-range integrity)
```sql
{QUERY_CHECK1A_REF_RANGE}
```

### Check 1b (quarter-label consistency)
```sql
{QUERY_CHECK1B_QUARTER_RANGES}
```
"""

md_path = ARTIFACTS / "01_05_08_leakage_audit.md"
md_path.write_text(md_content)
print("Wrote:", md_path)

print(f"\n# Verdict: {'PASS — no temporal leakage detected.' if all_pass else 'FAIL — investigate above.'}")
print("Done.")

Wrote: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/05_temporal_panel_eda/01_05_08_leakage_audit.md

# Verdict: PASS — no temporal leakage detected.
Done.
